# **Financial Month-End Closing Automation System**


## 第0步：安裝套件

In [1]:
# Google 官方的 Gemini SDK
!pip install -q -U google-genai

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from google import genai
from google.colab import userdata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.3/806.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.52.0 which is incompatible.
google-cloud-aiplatform 1.148.1 requires google-genai<2.0.0,>=1.66.0; python_version >= "3.10", but you have google-genai 2.2.0 which is incompatible.
google-adk 1.29.0 requires google-genai<2.0.0,>=1.64.0, but you have google-genai 2.2.0 which is incompatible.


## 第0步：串連api

In [2]:
# Colab 左側鑰匙圖示新增API金鑰
# 建立 Gemini client，client 連線 Gemini API，後續可呼叫模型gemini-3-flash-preview

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

## 第1步：模擬可能的場景，生成模擬 Excel 資料

In [3]:
def generate_large_scale_data():
    print("正在生成模擬 Excel 檔案...")

    depts = [
        {
            "name": "Sales",
            "file": "sales_large.xlsx",
            "date_col": "Order_Date",
            "amt_col": "Revenue",
            "dept_col": "Dept_Name",
        },
        {
            "name": "行銷部",
            "file": "marketing_large.xlsx",
            "date_col": "日期",
            "amt_col": "金額",
            "dept_col": "部門",
        },
        {
            "name": "管理部",
            "file": "admin_large.xlsx",
            "date_col": "Date",
            "amt_col": "Amount",
            "dept_col": "Dept",
        },
    ]

    np.random.seed(42)

    for d in depts:
        rows = 333

        # 產生4月的日期
        start_date = datetime(2026, 4, 1)
        random_days = np.random.randint(0, 30, size=rows)
        dates = [start_date + timedelta(days=int(i)) for i in random_days]

        # 產生一堆隨機金額，大部分集中在 20,000 附近
        amounts = np.random.normal(loc=20000, scale=15000, size=rows).astype(int)

        df = pd.DataFrame({
            d["date_col"]: dates,
            d["dept_col"]: [d["name"]] * rows,
            d["amt_col"]: amounts,
        })

        # 注入異常資料 故意讓 1% 的資料變成：負數金額、超大金額
        df.loc[df.sample(frac=0.01, random_state=1).index, d["amt_col"]] = -500
        df.loc[df.sample(frac=0.01, random_state=2).index, d["amt_col"]] = 250000

        # 行銷部還多一個異常 故意把部分資料日期改成 5 月，讓它變成「非本月資料」
        if d["name"] == "行銷部":
            df.loc[df.sample(frac=0.01, random_state=3).index, d["date_col"]] = datetime(2026, 5, 20)

        df.to_excel(d["file"], index=False)

    print("模擬 Excel 檔案生成成功！")

## 第2步：財務資料整併與異常檢查

In [4]:
# 批次讀取資料夾中的 Excel，統一欄位名稱，合併成總表，再檢查異常資料
def process_finance_logic(folder_path):

    # 用來暫時存放每一個 Excel 讀進來的資料表
    all_data = []

    # 雖然不同部門欄位名稱不一樣，但我把它們統一成：Date、Dept、Amt
    field_map = {
        "Date": ["Order_Date", "日期", "Date"],
        "Dept": ["Dept_Name", "部門", "Dept"],
        "Amt": ["Revenue", "金額", "Amount"],
    }
    # 建立欄位對照表，方便後面 rename 使用
    inv_map = {
        old_name: standard_name
        for standard_name, old_names in field_map.items()
        for old_name in old_names
    }

    # 它會去指定資料夾找出：檔名包含 large 副檔名是 .xlsx
    files = [
        f for f in os.listdir(folder_path)
        if "large" in f and f.endswith(".xlsx")
    ]
    # 每讀一個 Excel，就做四件事：讀檔案、統一欄位名稱、加上來源檔名、只留下需要的欄位
    for file in files:
        df = pd.read_excel(os.path.join(folder_path, file))

        df = df.rename(columns=inv_map)

        df["Source"] = file

        all_data.append(df[["Date", "Dept", "Amt", "Source"]])


    # 把三個部門的 Excel 疊成一張大表
    full_df = pd.concat(all_data, ignore_index=True)

    # 轉換資料型態，錯誤格式會轉成 NaN
    full_df["Amt"] = pd.to_numeric(full_df["Amt"], errors="coerce")
    full_df["Date"] = pd.to_datetime(full_df["Date"], errors="coerce")

    # 建立異常檢查欄位
    full_df["Check"] = ""

    # 進行異常資料條件檢查
    full_df.loc[full_df["Amt"] < 0, "Check"] += "金額為負數; "
    full_df.loc[full_df["Amt"] > 100000, "Check"] += "超過預算閾值; "
    full_df.loc[full_df["Date"].dt.month != 4, "Check"] += "非本月資料; "
    full_df.loc[full_df["Amt"].isna(), "Check"] += "金額格式錯誤; "
    full_df.loc[full_df["Date"].isna(), "Check"] += "日期格式錯誤; "

    # 篩選出異常資料
    anomaly_df = full_df[full_df["Check"] != ""].copy()

    return full_df, anomaly_df

## 第3步：生成Gemini AI 摘要

In [5]:
def get_ai_report_with_gemini(anomaly_df):

    # 異常資料表anomaly_df如果是空的
    if anomaly_df.empty:
        return "本月財務核銷無異常，所有部門報表格式與金額均符合規範。"

    total_anomalies = len(anomaly_df)
    dept_summary = anomaly_df["Dept"].value_counts().to_string()

    # 避免資料太多，先給 AI 前 10 筆代表性異常
    anomaly_sample = anomaly_df.head(10).to_string(index=False)

    prompt = f"""
你是金控財務部門實習生，請根據以下「月結異常資料」產生一份專業摘要。

請遵守：
1. 不要編造不存在的資料。
2. 只能根據我提供的異常清單摘要。
3. 語氣要像給財務部主管看的報告。
4. 請用繁體中文。
5. 請包含：
   - 總體異常概況
   - 各部門異常分布
   - 需要優先複核的項目
   - 後續處理建議

本月異常總筆數：{total_anomalies}

各部門異常筆數：
{dept_summary}

異常資料範例：
{anomaly_sample}
"""

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    return response.text

## 第4步：輸出 Excel 報表


In [6]:
def export_report(df_all, df_anomaly, ai_report):
    output_file = "finance_monthly_check_report.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_all.to_excel(writer, sheet_name="整併後總表", index=False)
        df_anomaly.to_excel(writer, sheet_name="異常清單", index=False)

        report_df = pd.DataFrame({
            "AI財務摘要": ai_report.split("\n")
        })
        report_df.to_excel(writer, sheet_name="AI摘要報告", index=False)

    print(f"已輸出 Excel 報表：{output_file}")

## 第5步：主執行流程

In [7]:
generate_large_scale_data()

print("正在處理資料...")
df_all, df_anomaly = process_finance_logic(".")

print(f"全部資料筆數：{len(df_all)}")
print(f"異常資料筆數：{len(df_anomaly)}")

if not df_anomaly.empty:
    print("\n--- 異常清單前幾筆 ---")
    print(df_anomaly.head())

print("\n--- 正在產生 Gemini AI 財務摘要 ---")
ai_report = get_ai_report_with_gemini(df_anomaly)

print("\n【Gemini AI 財務摘要報告】")
print(ai_report)

export_report(df_all, df_anomaly, ai_report)

正在生成模擬 Excel 檔案...
模擬 Excel 檔案生成成功！
正在處理資料...
全部資料筆數：999
異常資料筆數：102

--- 異常清單前幾筆 ---
         Date   Dept     Amt            Source     Check
6  2026-04-29  Sales   -4112  sales_large.xlsx   金額為負數; 
7  2026-04-21  Sales  250000  sales_large.xlsx  超過預算閾值; 
53 2026-04-25  Sales   -2722  sales_large.xlsx   金額為負數; 
59 2026-04-09  Sales    -500  sales_large.xlsx   金額為負數; 
68 2026-04-09  Sales    -665  sales_large.xlsx   金額為負數; 

--- 正在產生 Gemini AI 財務摘要 ---

【Gemini AI 財務摘要報告】
**呈：財務部主管**
**副本：財務部相關同仁**
**日期：2026年5月2日**
**主旨：2026年4月份月結異常資料分析摘要報告**

經針對本月月結資料進行系統自動檢核，初步篩選出共計 102 筆異常項目。現就異常概況、部門分布及重點風險項目彙整如下，請 鈞閱。

### 一、 總體異常概況
本月份月結檢核過程中，系統共偵測到 **102 筆** 異常帳務資料。相關異常主要集中於金額邏輯錯誤（如負數值）以及跨越預算控管閾值等類別。目前所有異常項均已完成初步分類，待各部門經辦進一步釐清原因。

### 二、 各部門異常分布
根據數據統計，異常筆數分布如下：
*   **行銷部**：44 筆（佔比約 43%），為本月異常頻次最高之部門。
*   **管理部**：30 筆（佔比約 29%）。
*   **業務部 (Sales)**：28 筆（佔比約 27%）。

儘管行銷部異常筆數最多，但業務部（Sales）之異常項目涉及多筆負值帳務，需額外關注其入帳邏輯。

### 三、 需要優先複核的項目
根據業務部（Sales）提供之 `sales_large.xlsx` 來源資料，以下兩類項目風險程度較高，建議優先進行人工覆核：

1